# dataloader-pin-memory-workers — faded example 3: Seed each worker's numpy RNG from torch.initial_seed

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `dataloader-pin-memory-workers`. Running the beacon reports progress on the `PyTorch: DataLoader pin_memory + workers` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: DataLoader pin_memory + workers` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dataloader-pin-memory-workers`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dataloader-pin-memory-workers"
DD_SUBTOPIC = "PyTorch: DataLoader pin_memory + workers"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Each DataLoader worker process inherits a distinct base seed accessible via `torch.initial_seed()`. To make per-worker augmentation reproducible you set `worker_init_fn` to seed numpy/`random` from that base offset by `worker_id`. Without this, every worker shares the same numpy seed and produces duplicated 'random' augmentations.

## Faded exercise 3

### Write the worker_init_fn that seeds numpy per worker

Implement `worker_init_fn(worker_id)` so that each worker seeds numpy's global RNG from `torch.initial_seed()` reduced modulo `2**32`, offset by `worker_id`. Then `make_seeded_loader` wires it into a `DataLoader` (with `num_workers=0` for notebook safety, so the function simply must be valid and callable).

Complete the blanked function body that computes the base seed and seeds numpy.

**Fill in:** Computes base = torch.initial_seed() % 2**32 and calls np.random.seed(base + worker_id).

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

def worker_init_fn(worker_id):
    raise NotImplementedError()  # TODO: seed numpy from torch.initial_seed() % 2**32 offset by worker_id

def make_seeded_loader(dataset, batch_size, num_workers):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=False,
        shuffle=True,
        worker_init_fn=worker_init_fn,
    )


def _test():
    # simulate the worker context: torch.initial_seed() is the manual seed in-main
    t.manual_seed(12345)
    base = t.initial_seed() % (2 ** 32)
    # calling worker_init_fn(wid) must seed numpy to (base + wid)
    worker_init_fn(3)
    got = np.random.rand(4)
    np.random.seed((base + 3) % (2 ** 32))
    expected = np.random.rand(4)
    assert np.allclose(got, expected), (got, expected)
    # different worker_id => different stream
    worker_init_fn(0)
    a = np.random.rand(4)
    worker_init_fn(1)
    b = np.random.rand(4)
    assert not np.allclose(a, b)
    # loader is constructible and wired
    ds = TensorDataset(t.arange(20, dtype=t.float32).reshape(20, 1))
    loader = make_seeded_loader(ds, batch_size=4, num_workers=0)
    assert loader.worker_init_fn is worker_init_fn
    total = sum(b[0].shape[0] for b in loader)
    assert total == 20, total


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
from torch.utils.data import DataLoader, TensorDataset

def worker_init_fn(worker_id):
    base = t.initial_seed() % (2 ** 32)
    np.random.seed(base + worker_id)

def make_seeded_loader(dataset, batch_size, num_workers):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=False,
        shuffle=True,
        worker_init_fn=worker_init_fn,
    )
```
</details>